## 持久化

In [ ]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#### 时间穿梭

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from typing import TypedDict, Literal


class State(TypedDict):
    topic: str
    mode: str
    poem: str
    joke: str
    output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    output: str

def node_poem(state: InputState):
    poem_msg = HumanMessage(f"请写一首关于{state["topic"]}的五言绝句，不用解释内容")
    ai_msg = model.invoke([poem_msg])
    return {
        "poem": ai_msg.content,
    }

def node_joke(state: InputState):
    joke_msg = HumanMessage(f"请写一个关于{state["topic"]}的超短笑话")
    ai_msg = model.invoke([joke_msg])
    return {
        "joke": ai_msg.content,
    }

def route(state: State) -> Literal["node_poem", "node_joke", "node_output"]:
    if state["mode"] == "poem":
        return "node_poem"
    elif state["mode"] == "joke":
        return "node_joke"
    else:
        raise "node_output"


def node_output(state: State):
    if state["mode"] == "poem":
        return {
            "output": f"主题：{state["topic"]}\n\n五言绝句：{state["poem"]}"
        }
    elif state["mode"] == "joke":
        return {
            "output": f"主题：{state["topic"]}\n\n笑话：{state["joke"]}"
        }
    else:
        return {"output": f"主题：{state["topic"]}\n\n不支持的模式"}


builder = StateGraph(State)
builder.add_node(node_poem)
builder.add_node(node_joke)
builder.add_node(node_output)
builder.add_conditional_edges(START, route)
builder.add_edge("node_poem", "node_output")
builder.add_edge("node_joke", "node_output")
builder.add_edge("node_output", END)
graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "4-4"}}
result = graph.invoke({"topic": "猫", "mode": "poem"}, config=config)
print(result["output"])

##### 1. Replay

In [ ]:
# 找到重放的位置
history = list(graph.get_state_history(config))
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")
before_poem = next(s for s in history if s.next == ("node_poem",))

# 执行重放
replay_result = graph.invoke(None, before_poem.config)

# 查询重放的内容
print("\n" + "-"*50 + "\n")
history = list(graph.get_state_history(config))
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

##### 2. Fork

In [ ]:
# 找到分叉的位置
history = list(graph.get_state_history(config))
before_poem = next(s for s in history if s.next == ("node_poem",))

# 更新状态并执行
fork_config = graph.update_state(before_poem.config, {"mode": "joke"})
fork_result = graph.invoke(None, fork_config)
print(fork_result["output"])

# 查询分叉的内容
history = list(graph.get_state_history(config))
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}")

In [ ]:
from rich.pretty import pprint

history_checkpoint = list(graph.get_state_history(config=config))
pprint(history_checkpoint)